# Equations
### Local (disk surface)
$$
\mathrm{St}_{s,S}=\frac{\pi}{2}\frac{\rho_{\rm int}a_s}{\Sigma_g}\exp\!\left(\frac{z^2}{2H_g^2}\right)
$$

$$
\beta_{\rm rad,d}=\frac{c_d\Omega}{16\,\sigma_{\rm SB}\kappa_{P,d}T_{\rm eq}^3},\quad
\beta_{\rm rad,g}=\frac{c_p\Omega}{16\,\sigma_{\rm SB}\kappa_{P,d}(\rho_d/\rho_g)T_{\rm eq}^3}
$$

$$
\beta_{\rm col,d}=\alpha_T^{-1}\frac{2}{3}\frac{\gamma_g}{\gamma_g-1}\frac{c_d}{c_p}\,\mathrm{St}_{s,S},\quad
\beta_{\rm col,g}=\alpha_T^{-1}\frac{2}{3}\frac{\gamma_g}{\gamma_g-1}\frac{\rho_g}{\rho_d}\,\mathrm{St}_{s,S}
$$

### Vertically integrated (disk interior)

$$
\tau_{\rm eff}=\frac{3\tau_{R,d}}{8}+\frac{\sqrt{3}}{4}+\frac{1}{4\tau_{P,d}},\quad
\tau_{R,d}=\frac{\kappa_{R,d}\Sigma_d}{2},\quad
\tau_{P,d}=\frac{\kappa_{P,d}\Sigma_d}{2}
$$

$$
\mathrm{St}_{\rm mid,S}=\frac{\pi}{2}\frac{a_s\rho_{\rm int}}{\Sigma_g}
$$

$$
\beta_{\rm rad,d}=\frac{\Sigma_d c_d\tau_{\rm eff}\Omega}{8\sigma_{\rm SB}T_{\rm eq}^3},\quad
\beta_{\rm rad,g}=\frac{\Sigma_g c_p\tau_{\rm eff}\Omega}{8\sigma_{\rm SB}T_{\rm eq}^3}
$$

$$
\beta_{\rm col,d}=\alpha_T^{-1}\frac{2}{3}\frac{\gamma_g}{\gamma_g-1}\frac{c_d}{c_p}\sqrt{1+(H_d/H_g)^2}\,\mathrm{St}_{\rm mid,S}
$$

$$
\beta_{\rm col,g}=\alpha_T^{-1}\frac{2}{3}\frac{\gamma_g}{\gamma_g-1}\frac{\Sigma_g}{\Sigma_d}\sqrt{1+(H_d/H_g)^2}\,\mathrm{St}_{\rm mid,S}
$$

In [1]:
import numpy as np

# Minimal notebook-ready script to explore cooling timescales vs physical inputs


# --- Constants (cgs) ---
sigma_SB = 5.670374419e-5   # Stefan-Boltzmann constant [erg cm^-2 s^-1 K^-4]
G = 6.67430e-8              # Gravitational constant [cm^3 g^-1 s^-2]
M_sun = 1.98847e33          # Solar mass [g]
AU = 1.495978707e13         # Astronomical unit [cm]

In [2]:
# Scenario 1: Local cooling at the disk surface

def keplerian_omega(r_AU, M_star_Msun):
    r = r_AU * AU
    M = M_star_Msun * M_sun
    return np.sqrt(G * M / r**3)

def local_surface_cooling(
    a_s, rho_int, Sigma_g, z_over_Hg, c_d, c_p, kappa_P_d, T_eq,
    gas_dust_ratio_vol, alpha_T, gamma_g, Omega
):
    # gas_dust_ratio_vol = rho_g / rho_d
    dust_to_gas_ratio_vol = 1.0 / gas_dust_ratio_vol

    St_s_S = (np.pi / 2.0) * (rho_int * a_s / Sigma_g) * np.exp(0.5 * z_over_Hg**2)

    beta_rad_d = (c_d * Omega) / (16.0 * sigma_SB * kappa_P_d * T_eq**3)
    beta_rad_g = (c_p * Omega) / (16.0 * sigma_SB * kappa_P_d * dust_to_gas_ratio_vol * T_eq**3)

    pref = (1.0 / alpha_T) * (2.0 / 3.0) * (gamma_g / (gamma_g - 1.0))
    beta_col_d = pref * (c_d / c_p) * St_s_S
    beta_col_g = pref * gas_dust_ratio_vol * St_s_S

    return {
        "St_s_S": St_s_S,
        "beta_rad_d": beta_rad_d,
        "beta_rad_g": beta_rad_g,
        "beta_col_d": beta_col_d,
        "beta_col_g": beta_col_g,
    }

# --- Inputs for disk surface scenario (relevant only) ---
surface_inputs = dict(
    r_AU=50.0,
    M_star_Msun=1.0,
    a_s=1e-3,                     # grain size [cm]
    rho_int=1.0,                 # grain internal density [g cm^-3]
    Sigma_g=100.0,               # gas surface density [g cm^-2]
    z_over_Hg=3.0,               # height in units of H_g
    c_d=1e7,                     # dust heat capacity [erg g^-1 K^-1]
    c_p=1e8,                     # gas heat capacity [erg g^-1 K^-1]
    kappa_P_d=10.0,              # Planck mean opacity [cm^2 g^-1]
    T_eq=100.0,                  # equilibrium temperature [K]
    gas_dust_ratio_vol=100.0,    # rho_g / rho_d
    alpha_T=1,                # thermal coupling parameter
    gamma_g=1.4,                 # adiabatic index
)

Omega_surface = keplerian_omega(surface_inputs["r_AU"], surface_inputs["M_star_Msun"])
surface_out = local_surface_cooling(
    a_s=surface_inputs["a_s"],
    rho_int=surface_inputs["rho_int"],
    Sigma_g=surface_inputs["Sigma_g"],
    z_over_Hg=surface_inputs["z_over_Hg"],
    c_d=surface_inputs["c_d"],
    c_p=surface_inputs["c_p"],
    kappa_P_d=surface_inputs["kappa_P_d"],
    T_eq=surface_inputs["T_eq"],
    gas_dust_ratio_vol=surface_inputs["gas_dust_ratio_vol"],
    alpha_T=surface_inputs["alpha_T"],
    gamma_g=surface_inputs["gamma_g"],
    Omega=Omega_surface, 
)

print("Scenario 1: Disk surface")
print("Omega [s^-1]:", Omega_surface)
for k, v in surface_out.items():
    print(f"  {k:16s} = {v:.6e}")

Scenario 1: Disk surface
Omega [s^-1]: 5.631437374853379e-10
Sigma_g(R) [g cm^-2]: 100.0
  St_s_S           = 1.413986e-03
  beta_rad_d       = 6.207083e-07
  beta_rad_g       = 6.207083e-04
  beta_col_d       = 3.299300e-04
  beta_col_g       = 3.299300e-01


In [3]:
# Scenario 2: Vertically integrated cooling in the disk interior

def keplerian_omega(r_AU, M_star_Msun):
    r = r_AU * AU
    M = M_star_Msun * M_sun
    return np.sqrt(G * M / r**3)

def vertical_integrated_cooling(
    a_s, rho_int, Sigma_g, gas_dust_ratio_vert_int, kappa_R_d, kappa_P_d, c_d, c_p, T_eq,
    alpha_T, gamma_g, Hd_over_Hg, Omega
):
    # gas_dust_ratio_vert_int = Sigma_g / Sigma_d
    Sigma_d = Sigma_g / gas_dust_ratio_vert_int

    tau_R_d = 0.5 * kappa_R_d * Sigma_d
    tau_P_d = 0.5 * kappa_P_d * Sigma_d
    tau_eff = (3.0 * tau_R_d / 8.0) + (np.sqrt(3.0) / 4.0) + (1.0 / (4.0 * tau_P_d))

    St_mid_S = (np.pi / 2.0) * (a_s * rho_int / Sigma_g)

    beta_rad_d = (Sigma_d * c_d * tau_eff * Omega) / (8.0 * sigma_SB * T_eq**3)
    beta_rad_g = (Sigma_g * c_p * tau_eff * Omega) / (8.0 * sigma_SB * T_eq**3)

    pref = (1.0 / alpha_T) * (2.0 / 3.0) * (gamma_g / (gamma_g - 1.0))
    geom = np.sqrt(1.0 + Hd_over_Hg ** 2)
    beta_col_d = pref * (c_d / c_p) * geom * St_mid_S
    beta_col_g = pref * gas_dust_ratio_vert_int * geom * St_mid_S

    return {
        "Sigma_d": Sigma_d,
        "tau_R_d": tau_R_d,
        "tau_P_d": tau_P_d,
        "tau_eff": tau_eff,
        "St_mid_S": St_mid_S,
        "beta_rad_d": beta_rad_d,
        "beta_rad_g": beta_rad_g,
        "beta_col_d": beta_col_d,
        "beta_col_g": beta_col_g,
    }

# --- Inputs for disk interior scenario (relevant only) ---
interior_inputs = dict(
    r_AU=50.0,
    M_star_Msun=1.0,
    a_s=1e-1,                     # grain size [cm]
    rho_int=1.0,                  # grain internal density [g cm^-3]
    Sigma_g=100.0,                # gas surface density [g cm^-2]
    gas_dust_ratio_vert_int=100.0,  # Sigma_g / Sigma_d
    kappa_R_d=10.0,               # Rosseland mean opacity [cm^2 g^-1]
    kappa_P_d=10.0,               # Planck mean opacity [cm^2 g^-1]
    c_d=1e7,                      # dust heat capacity [erg g^-1 K^-1]
    c_p=1e8,                      # gas heat capacity [erg g^-1 K^-1]
    T_eq=30.0,                    # equilibrium temperature [K]
    alpha_T=1,                    # thermal coupling parameter
    gamma_g=1.4,                  # adiabatic index
    Hd_over_Hg=0.2,               # H_d / H_g
)

Omega_interior = keplerian_omega(interior_inputs["r_AU"], interior_inputs["M_star_Msun"])
interior_out = vertical_integrated_cooling(
    a_s=interior_inputs["a_s"],
    rho_int=interior_inputs["rho_int"],
    Sigma_g=interior_inputs["Sigma_g"],
    gas_dust_ratio_vert_int=interior_inputs["gas_dust_ratio_vert_int"],
    kappa_R_d=interior_inputs["kappa_R_d"],
    kappa_P_d=interior_inputs["kappa_P_d"],
    c_d=interior_inputs["c_d"],
    c_p=interior_inputs["c_p"],
    T_eq=interior_inputs["T_eq"],
    alpha_T=interior_inputs["alpha_T"],
    gamma_g=interior_inputs["gamma_g"],
    Hd_over_Hg=interior_inputs["Hd_over_Hg"],
    Omega=Omega_interior, 
)

print("Scenario 2: Disk interior")
print("Omega [s^-1]:", Omega_interior)
for k, v in interior_out.items():
    print(f"  {k:16s} = {v:.6e}")

Scenario 2: Disk interior
Omega [s^-1]: 5.631437374853379e-10
Sigma_g(R) [g cm^-2]: 100.0
  Sigma_d          = 1.000000e+00
  tau_R_d          = 5.000000e+00
  tau_P_d          = 5.000000e+00
  tau_eff          = 2.358013e+00
  St_mid_S         = 1.570796e-03
  beta_rad_d       = 1.084176e-03
  beta_rad_g       = 1.084176e+00
  beta_col_d       = 3.737777e-04
  beta_col_g       = 3.737777e-01
